# Day 47：实战场景——Text-to-SQL 企业数据分析智能体

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第四阶段（Day 47）！今天我们将基于 `v0.4.0-day47` 实战演示 **Text-to-SQL 数据分析智能体**：从真实 SQLite 销售业务库 **`SalesDatabase`** 建表与 Schema 自省，到只读 SQL 安全护栏 **`validate_readonly_sql()`** 防御注入攻击，再到 **`TextToSQLAgent`** 完成多表 `JOIN` 问数与中文商业洞察摘要。

## 1. 环境初始化与导入 `project_iterations/day_47` 模块
定位仓库根目录并将 `project_iterations/day_47` 加入 `sys.path`。

In [1]:
import json
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day47_dir = repo_root / "project_iterations" / "day_47"
if str(day47_dir) not in sys.path:
    sys.path.insert(0, str(day47_dir))

from zhishu_agent import (
    SQLSecurityError,
    SalesDatabase,
    TextToSQLAgent,
    validate_readonly_sql,
)

print("已加载模块路径:", day47_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_47


## 2. 初始化 `SalesDatabase` 与 Schema 自省
创建包含 `regions`、`products`、`orders` 三张关联表的 SQLite 数据库并写入示例销售数据，查看 `introspect_schema()` 输出。

In [2]:
db = SalesDatabase(":memory:")
db.initialize_and_seed()

print("数据表列表:", db.get_table_names())
print(db.introspect_schema())

数据表列表: ['orders', 'products', 'regions']
数据库表结构（SQLite）：
- 表 `orders`（12 行）: CREATE TABLE orders ( order_id INTEGER PRIMARY KEY, product_id INTEGER NOT NULL, region_id INTEGER NOT NULL, customer_name TEXT NOT NULL, quantity INTEGER NOT NULL, total_amount REAL NOT NULL, order_date TEXT NOT NULL, status TEXT NOT NULL, FOREIGN KEY (product_id) REFERENCES products(product_id), FOREIGN KEY (region_id) REFERENCES regions(region_id) )
- 表 `products`（5 行）: CREATE TABLE products ( product_id INTEGER PRIMARY KEY, product_name TEXT NOT NULL UNIQUE, category TEXT NOT NULL, unit_price REAL NOT NULL )
- 表 `regions`（4 行）: CREATE TABLE regions ( region_id INTEGER PRIMARY KEY, region_name TEXT NOT NULL UNIQUE, manager TEXT NOT NULL, target_revenue REAL NOT NULL )


## 3. 只读 SQL 安全护栏（`validate_readonly_sql`）攻防测试
验证合法 `SELECT` 语句正常放行，而 `DROP`、`DELETE`、`UPDATE`、`INSERT`、`ALTER`、`PRAGMA` 与分号 `;` 堆叠注入被严格拦截。

In [3]:
safe_sql = "SELECT region_name, target_revenue FROM regions ORDER BY target_revenue DESC"
print("合法 SQL 校验通过:", validate_readonly_sql(safe_sql))

attack_payloads = [
    "DROP TABLE orders",
    "DELETE FROM orders WHERE order_id = 1001",
    "UPDATE orders SET total_amount = 0",
    "INSERT INTO regions VALUES (9, '测试区', 'Test', 100)",
    "ALTER TABLE orders ADD COLUMN hack TEXT",
    "PRAGMA table_info(orders)",
    "SELECT * FROM orders; DROP TABLE orders",
]

for payload in attack_payloads:
    try:
        validate_readonly_sql(payload)
        raise AssertionError(f"未拦截危险语句: {payload}")
    except SQLSecurityError as exc:
        print(f"[已拦截] {payload} -> {exc}")

合法 SQL 校验通过: SELECT region_name, target_revenue FROM regions ORDER BY target_revenue DESC
[已拦截] DROP TABLE orders -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] DELETE FROM orders WHERE order_id = 1001 -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] UPDATE orders SET total_amount = 0 -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] INSERT INTO regions VALUES (9, '测试区', 'Test', 100) -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] ALTER TABLE orders ADD COLUMN hack TEXT -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] PRAGMA table_info(orders) -> 仅允许以 SELECT 开头的只读查询语句
[已拦截] SELECT * FROM orders; DROP TABLE orders -> 检测到非法分号 ';'，禁止多语句执行或堆叠注入


## 4. `TextToSQLAgent` 端到端多表 `JOIN` 问数与商业洞察
演示从自然语言提问到 SQL 生成、安全执行与中文商业洞察摘要的完整闭环，并测试恶意 `llm_fn` 注入防护。

In [4]:
agent = TextToSQLAgent(db=db)

questions = [
    "各大区已完成订单的实际销售额与目标达成率是多少？",
    "销售额排名前 5 的产品有哪些？",
    "按产品类别统计销售额和订单数",
]

for q in questions:
    res = agent.ask(q)
    assert res.status == "success"
    print(f"问题: {q}")
    print(f"SQL : {res.generated_sql}")
    print(f"结果: {json.dumps(res.rows[:3], ensure_ascii=False)}")
    print(f"洞察: {res.insight}\n")

# 模拟上游 LLM 被诱导输出堆叠注入 SQL
guarded_agent = TextToSQLAgent(db=db, llm_fn=lambda q, s: "SELECT * FROM orders; DROP TABLE orders")
blocked_res = guarded_agent.ask("帮我删掉订单表")
assert blocked_res.status == "blocked"
print("恶意 LLM 输出拦截状态:", blocked_res.status, "|", blocked_res.insight)

db.close()

问题: 各大区已完成订单的实际销售额与目标达成率是多少？
SQL : SELECT r.region_name AS 大区, r.manager AS 负责人, ROUND(SUM(o.total_amount), 2) AS 实际销售额, r.target_revenue AS 目标销售额, ROUND(SUM(o.total_amount) * 100.0 / r.target_revenue, 1) AS 达成率百分比 FROM orders o JOIN regions r ON o.region_id = r.region_id WHERE o.status = 'completed' GROUP BY r.region_id, r.region_name, r.manager, r.target_revenue ORDER BY 实际销售额 DESC
结果: [{"大区": "华东", "负责人": "陈敏", "实际销售额": 240000.0, "目标销售额": 200000.0, "达成率百分比": 120.0}, {"大区": "华南", "负责人": "林峰", "实际销售额": 152000.0, "目标销售额": 160000.0, "达成率百分比": 95.0}, {"大区": "华北", "负责人": "张伟", "实际销售额": 96000.0, "目标销售额": 150000.0, "达成率百分比": 64.0}]
洞察: 商业洞察：共统计 4 个大区已完成订单。表现最佳为【华东】（负责人：陈敏），实际销售额 240,000 元，目标达成率 120.0%；相对滞后为【华北】，达成率 64.0%，建议重点跟进大客户转化。

问题: 销售额排名前 5 的产品有哪些？
SQL : SELECT p.product_name AS 产品名称, p.category AS 类别, SUM(o.quantity) AS 销量, ROUND(SUM(o.total_amount), 2) AS 销售额 FROM orders o JOIN products p ON o.product_id = p.product_id GROUP BY p.product_id, p.product_name, p.category ORDER BY 销售额 